In [ ]:
from pathlib import Path
import os
import sys

# Make paths and `src` imports work whether Jupyter starts in the repository root
# or in the notebooks directory.
PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "src").is_dir():
    PROJECT_ROOT = next(
        (parent for parent in PROJECT_ROOT.parents if (parent / "src").is_dir()),
        None,
    )
if PROJECT_ROOT is None:
    raise RuntimeError("Could not locate the repository root containing src/")
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.plotting import plot_unmasked_placepulse_heatmap, plot_sequence_stack, plot_big_heatmap, impute_equivalents, plot_phi_significance, phi_legend_handles
from src.shap_analysis import pivot_matrix, build_phi, build_delta_phi
from matplotlib.colors import LinearSegmentedColormap
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from pathlib import Path
import glob

In [ ]:
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point, LineString
import matplotlib.pyplot as plt
import contextily as cx 

sequences = [
    '3HYULmaTgG8jsDdSW1xKnc',
    '6yPCl23wumovTtV5giQN8L',
    'eyygIIVTe_eevi-HprCEow',
    'HEoA5tNxsvPYQ1h0k46m2D',
    'jEb1lhO6caSX20KHiT8zJw',
]

metadata = []
for sequence in sequences:
    seq_df = pd.read_csv(f'images/{sequence}/metadata.csv')
    image_files =  [Path(p).name for p in glob.glob(f'images/{sequence}/*.jpg')]
    seq_df = seq_df[seq_df.image_file.isin(image_files)]
    metadata.append(seq_df)
    
colors = ["#3B4EB6", "#F14272", "#15CA7A", "#ECACBE", "#ACB4E2"]
fig, ax = plt.subplots(figsize=(10,10))

for i, df in enumerate(metadata):
    df['geometry'] = [Point(x,y) for x,y in zip(df['lon'].values, df['lat'].values)]

    gdf = gpd.GeoDataFrame(df, geometry='geometry', crs='WGS84')
    gdf = gdf.to_crs(epsg=3857)
    gdf.plot(ax=ax, color=colors[i], label= f"Sequence {i}")

ax.set_xlim(-486000, -466000)
ax.set_axis_off()
ax.legend()
cx.add_basemap(ax, source=cx.providers.CartoDB.Positron)

plt.savefig('plots/sequence_map.pdf', bbox_inches='tight', dpi=300)
plt.savefig('plots/sequence_map.png', bbox_inches='tight', dpi=300)

# 1. Per sequence model outputs
The section will explore the basic properties of the scores given to each image within a seqence $Q$, across the PlacePulse dimensions $p \in \mathcal{P}$. This will include a panel plot of the proportion of each image taken up by a category ($\pi_c(I)$) and the score given to an image for a dimension $p$,  $f_p(I)$.  We will also calculate the mean 
 score $\mu_f^Q$ across images and the standard deviation $\sigma_f^Q$ as a measure of the degree of instability in the scores.
 
## 1.1 Panel Plot
**NOTE**: The call to "impute equivalents" is important here. For images where not all categories are present, we assume that the score for images with missing categories masked can be filled in from other scores. For example, if "person" was missing from an image, the score for the image with "vegetation", "building" and "person" masked can be assumed to be equal to the score with "vegetation" and "building" masked only. This saves considerable compute time, halving the number of model calls per missing category.

In [ ]:


def paste_fig_into_axes(fig, ax):
    buf = io.BytesIO()
    fig.savefig(buf, format='png', dpi=200, bbox_inches='tight')
    buf.seek(0)
    im = np.array(Image.open(buf))
    ax.imshow(im)
    ax.axis('off')

sequences = [
    '3HYULmaTgG8jsDdSW1xKnc',
    '6yPCl23wumovTtV5giQN8L',
    'eyygIIVTe_eevi-HprCEow',
    'HEoA5tNxsvPYQ1h0k46m2D',
    'jEb1lhO6caSX20KHiT8zJw',
]
category_order = ['sky', 'street', 'building', 'vegetation', 'traffic', 'person']

prop_cmap = ["#93a3fa", "#3B4EB6", "#f589a7", "#F14272", "#D8D8D8", "#444444"]
score_cmap = LinearSegmentedColormap.from_list("", ["#3B4EB6", "#ffffff", "#F14272"])

# --- compose the grid ---------------------------------------------------
n_rows = len(sequences)
fig, axes = plt.subplots(n_rows, 2, figsize=(12, 2.6*n_rows), constrained_layout=True)

# keep references for a shared legend/colorbar
category_handles_labels = None
first_heatmap_mappable = None

unmasked_dfs = {}
for i, seq in enumerate(sequences):
    # LEFT: stacked proportions
    meta_csv = f'images/masks/{seq}/metadata.csv'
    ax_left = axes[i, 0]
    # ideally your function supports: plot_sequence_stack(..., ax=ax_left, legend=False)
    prop_fig, prop_ax, grouped, *rest = plot_sequence_stack(
        meta_csv,
        category_order=category_order,
        cmap=prop_cmap,
        ax=ax_left,                # <-- add in your function
        legend=False               # <-- suppress per-plot legends
    )
    if category_handles_labels is None:
        # try to grab legend info just once
        handles, labels = ax_left.get_legend_handles_labels()
        if labels:
            category_handles_labels = (handles, labels)

    # RIGHT: heatmap of scores
    res_csv = f'results/{seq}.csv'
    ax_right = axes[i, 1]
    # ideally your function supports: plot_unmasked_placepulse_heatmap(..., ax=ax_right, add_colorbar=False)
    score_fig, score_ax, dims_used, df_unmasked, *rest2 = plot_unmasked_placepulse_heatmap(
        res_csv,
        cmap=score_cmap,
        ax=ax_right,               # <-- add in your function
        add_colorbar=False         # <-- we'll add a shared one
    )
    # keep a copy of the unmasked_df for later analysis
    unmasked_dfs.update({seq:df_unmasked})
    # capture the mappable once (for shared colorbar)
    if first_heatmap_mappable is None:
        # try to find an AxesImage/QuadMesh on the axis
        if ax_right.images:
            first_heatmap_mappable = ax_right.images[0]
        elif ax_right.collections:
            first_heatmap_mappable = ax_right.collections[0]

    # row labels / tidy bits
    ax_left.set_ylabel(f"Sequence {i+1} \n $\pi_c(I)$", rotation=90)
    if i == 0:
        ax_left.set_title("Category proportions")
        ax_right.set_title("Dimension scores")
        
        
n_rows = len(sequences)

# hide x tick *labels* on all but the last row

for r in range(n_rows):
    for c in range(2):
        axes[r, c].set_xlabel('')                 # remove label text
        #axes[r, c].xaxis.label.set_visible(False) # belt-and-braces
        axes[r, c].tick_params(labelbottom=True)  # ensure tick labels are shown

# only the bottom row shows per-axis xlabels
axes[-1,0].set_xlabel("Image index")
axes[-1,1].set_xlabel("Image index")

# (optional) one shared xlabel instead of per-axis labels:
#fig.supxlabel("Image index")
  
left_top   = axes[0, 0].get_position()
right_top  = axes[0, 1].get_position()
left_x0, left_x1   = left_top.x0, left_top.x1
right_x0, right_x1 = right_top.x0, right_top.x1

handles, labels = axes[0, 0].get_legend_handles_labels()

# Shared legend for categories (bottom center)
if category_handles_labels is not None:
    legend_x = left_x0 + 0.09    # adjust to taste
    legend_y = - 0.055              # inside the figure (no clipping)
    legend_h = 0.03              # legend height in fig coords

    fig.legend(
        handles, labels,
        ncol=3, frameon=True,
        loc='lower left',
        bbox_to_anchor=(legend_x, legend_y, left_x1 - left_x0 - 0.12, legend_h),
        mode='expand',
        title='Category'
    )

# Shared colorbar for score heatmaps (right side)
if first_heatmap_mappable is not None:
    cax = fig.add_axes([right_x0, legend_y+0.025, right_x1 - right_x0, legend_h])
    cbar = fig.colorbar(first_heatmap_mappable, cax=cax, orientation='horizontal', fraction=0.05)
    cbar.set_label("Score $f_p(I_{\\varnothing})$")

fig.tight_layout()
fig.savefig('plots/scores_and_props.pdf', bbox_inches='tight', dpi=300)
plt.show()

## 1.2 Stats

In [ ]:
# placepulse dimensions
place_pulse = ["safe","lively","beautiful","wealthy","depressing","boring"]
stats_data = []

for p in place_pulse:
    row = {'dimension':p}
    for i, seq in enumerate(sequences):
        data = unmasked_dfs[seq]
        row[f'seq_{i+1}_mu'] = np.mean(data[f'score_{p}'])
        row[f'seq_{i+1}_sigma'] = np.std(data[f'score_{p}'])
    stats_data.append(row)

stats_df = pd.DataFrame(stats_data)
        

In [ ]:
stats_df.to_csv('results/score_mean_std.csv', index=False)

# 2. Single Sequence Shapley scores
In this section we will plot the Shapley scores for sequence 3 only.

In [ ]:
sequence_id = 'eyygIIVTe_eevi-HprCEow'
df = pd.read_csv(f"results/{sequence_id}.csv")
# fill in an rows with empty categories
df = impute_equivalents(df, min_frac=0.01)
# fromat the labels
df["image_id"] = df["image_id"].astype(str)
df["label"] = df["label"].astype(str).str.lower()


phi_df = build_phi(df, seq_id=sequence_id)
piv = pivot_matrix(phi_df)  # reorder images by total |phi|

plot_big_heatmap(piv, out_png=f'plots/seq3_shapley_plot.pdf', title="Sequence $Q_3$ Shapely Values")

# 3. Presence Hallucinations
This section explores whether images marked with rational $R_p(I) \in \{NA, PA\}$ correspond to minimal or missing $pi_c(I)$ vlaues, and conversley $R_p(I) \in \{NP, PP\}$ correspond to higher values. Any deviation from these expectations would indicate potenital hallucinations, in which the model is identifying features whcih are not in the image. 

In [ ]:
import math 

pi_absent_q = {}
pi_present_q = {}
cats = ['building', 'traffic', 'street', 'vegetation', 'sky', 'person']
dims = ['safe', 'lively', 'beautiful', 'wealthy', 'depressing', 'boring']

## loop over sequences
for seq in sequences:
    # get unmasked images in sequence
    df = pd.read_csv(f"results/{seq}.csv", na_values=[''], keep_default_na=False)
    unmasked_df = df[df.label=='unmasked']
    # initialise storage vectors for sequence
    pi_absent_q[seq] = []
    pi_present_q[seq] = []
    # loop over features
    for c in cats:
        # initialise storage vecrots for feature
        pi_p = {c:np.nan}
        pi_a = {c:np.nan}
        for p in dims:
            m_pc_a = unmasked_df[unmasked_df[f'{p}_{c}_m'].isin(['PA','NA'])]
            m_pc_p = unmasked_df[unmasked_df[f'{p}_{c}_m'].isin(['PP','NP'])]
            
            if (len(m_pc_a) > 0) and (math.isnan(pi_a[c])):
                pi_a[c] = np.mean(m_pc_a[f'{c}_area_full'])
            if (len(m_pc_p) > 0) and (math.isnan(pi_p[c])):
                pi_p[c] = np.mean(m_pc_p[f'{c}_area_full'])
        pi_absent_q[seq].append(pi_a)
        pi_present_q[seq].append(pi_p)
            



In [ ]:
import numpy as np
import pandas as pd

def build_res_df(raw):

# ----- normalize to long dataframe -----
    records = []
    for seq_id, entries in raw.items():
        for d in entries:
            # each "d" is a single-key dict: {category: value}
            (category, value), = d.items()
            # coerce to float; treat unrecognized 'nan' safely
            try:
                val = float(value) if value is not None else np.nan
            except Exception:
                val = np.nan
            records.append({"seq_id": seq_id, "category": category, "value": val})

    df_long = pd.DataFrame(records)

    # optional: ensure category order (if you care about column order later)
    cat_order = ["building", "street", "vegetation", "sky", "traffic", "person"]
    df_long["category"] = pd.Categorical(df_long["category"], categories=cat_order, ordered=True)

    # ----- wide dataframe: one row per sequence, columns = categories -----
    df_wide = (df_long
            .pivot(index="seq_id", columns="category", values="value")
            .sort_index(axis=1))

    # ----- average across sequences for each category -----
    category_mean = df_wide.mean(axis=0, skipna=True).rename("mean_across_sequences")

    # (optional) also useful: number of non-nan sequences per category
    category_count = df_wide.notna().sum(axis=0).rename("n_sequences")

    # compact summary table
    summary = pd.concat([category_mean, category_count], axis=1)

    # If you want the mean appended as an extra row in the wide table:
    df_wide_with_mean = pd.concat(
        [df_wide, category_mean.to_frame().T],
        axis=0
    ).rename(index={"mean_across_sequences": "MEAN_ACROSS_SEQUENCES"})

    return df_long, df_wide, summary

absent_df_long, absent_df_wide, absent_summary = build_res_df(pi_absent_q)
present_df_long, present_df_wide, present_summary = build_res_df(pi_present_q)


In [ ]:
absent_df_wide

In [ ]:
present_df_wide

In [ ]:
absent_summary

In [ ]:
present_summary

# 3. Relative Shapley Support
This section explores whether, on average, images marked with $r_{p,c}=PP$, correspond to higher $\phi_{p,c}$ compared to equivalent images marked $r_{p,c} \in {NP, U}$. This will be evaluated over the entire dataset. The quantity $\Delta_{p,c}(x,y)$ refers to the difference between mean $\phi_{p,c}$ values when $r_{p,c} = x$ and $r_{p,c}=y$.  

In [ ]:
# first we build a combined dataframe full all images

full_data = []

for s in sequences:
    df = pd.read_csv(f'results/{s}.csv')
    df = impute_equivalents(df, min_frac=0.01)
    df["image_id"] = df["image_id"].astype(str)
    df["label"] = df["label"].astype(str).str.lower()
    phi_df = build_phi(df, seq_id=s)
    full_data.append(phi_df)

full_df = pd.concat(full_data)

# get the phi delta table
delta_df = build_delta_phi(full_df)


In [ ]:
# --- choose global symmetric limits across all panels (robust to outliers) ---
def global_vmax_for(df, comps):
    vals = []
    for comp in comps:
        slice_ = df.loc[df.comparison == comp, ["mean1", "mean2"]].astype(float)
        vals.append(np.abs((slice_["mean1"] - slice_["mean2"]).to_numpy()))
    return np.percentile(np.concatenate(vals), 95)

comps = ["PP vs NP", "PP vs U", "NP vs U"]
gmax = global_vmax_for(delta_df, comps)

# --- build figure ---
fig, axes = plt.subplots(3, 1, figsize=(6, 20), dpi=150)
fig.subplots_adjust(hspace=0.35, bottom=0.20)   # extra bottom space for legend + cbar

ims = []
ims.append(plot_phi_significance(
    delta_df[delta_df.comparison == "PP vs NP"],
    comparison=("PP", "NP"),
    features=['building','vegetation','street','sky'],
    vmin=-gmax, vmax=gmax, ax=axes[0],
    alpha=0.05, n_min=5,
)[0])

ims.append(plot_phi_significance(
    delta_df[delta_df.comparison == "PP vs U"],
    comparison=("PP", "U"),
    features=['building','vegetation','street','sky'],
    vmin=-gmax, vmax=gmax, ax=axes[1],
    alpha=0.05, n_min=10,
)[0])

ims.append(plot_phi_significance(
    delta_df[delta_df.comparison == "NP vs U"],
    comparison=("NP", "U"),
    features=['building','vegetation','street','sky'],
    vmin=-gmax, vmax=gmax, ax=axes[2],
    alpha=0.05, n_min=10,
)[0])

# --- one shared HORIZONTAL colorbar below all panels ---
# use the last image for its cmap/norm (all share vmin/vmax)
cbar = fig.colorbar(ims[-1], ax=axes, orientation="horizontal",
                    fraction=0.06, pad=0.05)  # tune pad/fraction as you like
cbar.set_label(r"$\Delta_{p,c}$", fontsize=12)
# nice symmetric ticks
ticks = np.linspace(-gmax, gmax, 5)
cbar.set_ticks(ticks)

# --- one horizontal legend below the colorbar ---
handles = phi_legend_handles(alpha=0.05)
fig.legend(handles=handles, ncol=4,
           loc="lower center", bbox_to_anchor=(0.5, 0.165),
           frameon=False, title="Direction and significance of $\Delta_{p,c}(A,B)$")

fig.savefig('plots/relative_phi_support.pdf', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Suppose you already have:
from src.plotting import compute_z_by_seq, threshold_sig_mask, plot_z_heatmap_textsig, z_legend_handles
features= ['building','vegetation','street','sky']
df4 = full_df[full_df["feature"].isin(features)].copy()

z_pp, nimg_pp, nseq_pp, _, _ = compute_z_by_seq(df4, "PP", alpha=0.05, fdr=False)
z_np, nimg_np, nseq_np, _, _ = compute_z_by_seq(df4, "NP", alpha=0.05, fdr=False)
z_u,  nimg_u,  nseq_u,  _, _ = compute_z_by_seq(df4, "U",  alpha=0.05, fdr=False)

sig_pp = threshold_sig_mask(z_pp, "PP", th_small=0.2, th_medium=0.5)
sig_np = threshold_sig_mask(z_np, "NP", th_small=0.2, th_medium=0.5)
sig_u  = threshold_sig_mask(z_u,  "U",  th_small=0.2, th_medium=0.5)

# Decide on features subset (same as your previous plot)
features_subset = ['building', 'vegetation', 'street', 'sky']

# Compute a global symmetric vmax across the three tabs (robust p95)
def _global_vmax(*tabs):
    vals = []
    for t in tabs:
        arr = np.asarray(t.to_numpy(dtype=float))
        arr = arr[np.isfinite(arr)]
        if arr.size:
            vals.append(np.abs(arr))
    if not vals:
        return 1.0
    return np.percentile(np.concatenate(vals), 95)

gmax = _global_vmax(z_pp, z_np, z_u)
vmin, vmax = -gmax, gmax

fig, axes = plt.subplots(3, 1, figsize=(6, 20), dpi=150)
fig.subplots_adjust(hspace=0.3, bottom=0.1)  # space for legend + colorbar

ims = []
ims.append(plot_z_heatmap_textsig(z_pp, nimg_pp, "PP",
                                  features=features_subset, vmin=vmin, vmax=vmax, ax=axes[0])[0])
ims.append(plot_z_heatmap_textsig(z_np, nimg_np, "NP",
                                  features=features_subset, vmin=vmin, vmax=vmax, ax=axes[1])[0])
ims.append(plot_z_heatmap_textsig(z_u,  nimg_u,  "U",
                                  features=features_subset, vmin=vmin, vmax=vmax, ax=axes[2])[0])

# Shared horizontal colorbar below panels
cbar = fig.colorbar(ims[-1], ax=axes, orientation="horizontal", fraction=0.06, pad=0.05)
cbar.set_label(r"mean $z$ (per-sequence normalised)", fontsize=12)
cbar.set_ticks(np.linspace(vmin, vmax, 5))

# Single horizontal legend below colorbar
#handles = z_legend_handles()
#fig.legend(handles=handles, ncol=4, loc="lower center", bbox_to_anchor=(0.5, 0),
#           frameon=False, title="Color & text key")

# Optional figure-level axis labels (consistent with your earlier convention)
#fig.text(0.5, 0.985, r'PlacePulse dimension $p$', ha='center', va='top', fontsize=12)
#fig.text(0.012, 0.5,  r'feature $c$', rotation=90, ha='left', va='center', fontsize=12)

plt.show()
fig.savefig('plots/within_sequence_phi.pdf', dpi=300, bbox_inches='tight')

## Masking Sensitivity
We have also run the VLM on a single sequence (`eyygIIVTe_eevi-HprCEow`) with different masking protocols - blur only (18 pixels), blur only (12 pixels), fully masked (white pixels). We will compare the Shapley scores with that of the blurred/grayscaled baseline. 

In [ ]:
## BASELINE
sequence_id = 'eyygIIVTe_eevi-HprCEow'
df = pd.read_csv(f"results/{sequence_id}.csv")
# fill in an rows with empty categories
df = impute_equivalents(df, min_frac=0.01)
# fromat the labels
df["image_id"] = df["image_id"].astype(str)
df["label"] = df["label"].astype(str).str.lower()


phi_df = build_phi(df, seq_id=sequence_id)
baseline_piv = pivot_matrix(phi_df)  # reorder images by total |phi|

plot_big_heatmap(baseline_piv, title="Sequence $Q_3$ Shapely Values: 18 Pixel Blur with Grayscale", out_png=f'plots/seq3_shapley_baseline_plot.pdf')

In [ ]:
## 18 Pixel Blur Only (No Grayscale)
sequence_id = 'eyygIIVTe_eevi-HprCEow_blur_18'
df = pd.read_csv(f"results/{sequence_id}.csv")
# fill in an rows with empty categories
df = impute_equivalents(df, min_frac=0.01)
# fromat the labels
df["image_id"] = df["image_id"].astype(str)
df["label"] = df["label"].astype(str).str.lower()


phi_df = build_phi(df, seq_id=sequence_id)
blur_18_piv = pivot_matrix(phi_df)  # reorder images by total |phi|

plot_big_heatmap(blur_18_piv, title="Sequence $Q_3$ Shapely Values: 18 pixel blur (no grayscale)", out_png=f'plots/seq3_shapley_blur18_plot.png')

In [ ]:
## 12 Pixel Blur Only (No Grayscale)
sequence_id = 'eyygIIVTe_eevi-HprCEow_blur_12'
df = pd.read_csv(f"results/{sequence_id}.csv")
# fill in an rows with empty categories
df = impute_equivalents(df, min_frac=0.01)
# fromat the labels
df["image_id"] = df["image_id"].astype(str)
df["label"] = df["label"].astype(str).str.lower()


phi_df = build_phi(df, seq_id=sequence_id)
blur_12_piv = pivot_matrix(phi_df)  # reorder images by total |phi|

plot_big_heatmap(blur_12_piv, title="Sequence $Q_3$ Shapely Values: 12 pixel blur (no grayscale)", out_png=f'plots/seq3_shapley_blur12_plot.png')

In [ ]:
## solid grey
sequence_id = 'eyygIIVTe_eevi-HprCEow_solid_grey'
df = pd.read_csv(f"results/{sequence_id}.csv")
# fill in an rows with empty categories
df = impute_equivalents(df, min_frac=0.01)
# fromat the labels
df["image_id"] = df["image_id"].astype(str)
df["label"] = df["label"].astype(str).str.lower()


phi_df = build_phi(df, seq_id=sequence_id)
solid_grey_piv = pivot_matrix(phi_df)  # reorder images by total |phi|

plot_big_heatmap(solid_grey_piv, title="Sequence $Q_3$ Shapely Values: solid grey", out_png=f'plots/seq3_shapley_solifd_grey_plot.png')

In [ ]:
## solid white
sequence_id = 'eyygIIVTe_eevi-HprCEow_solid_white'
df = pd.read_csv(f"results/{sequence_id}.csv")
# fill in an rows with empty categories
df = impute_equivalents(df, min_frac=0.01)
# fromat the labels
df["image_id"] = df["image_id"].astype(str)
df["label"] = df["label"].astype(str).str.lower()


phi_df = build_phi(df, seq_id=sequence_id)
solid_white_piv = pivot_matrix(phi_df)  # reorder images by total |phi|

plot_big_heatmap(solid_white_piv, title="Sequence $Q_3$ Shapely Values: solid white", out_png=f'plots/seq3_shapley_solid_white_plot.png')

In [ ]:
# NORMALISE PER DIMENSION
norm_blur12 = blur_12_piv.divide(blur_12_piv.max().max()).replace(0, np.nan)
norm_blur18 = blur_18_piv.divide(blur_18_piv.max().max()).replace(0, np.nan)
norm_baseline = baseline_piv.divide(baseline_piv.max().max()).replace(0, np.nan)

delta_blur18_baseline = norm_blur18 - norm_baseline
delta_blur12_baseline = norm_blur12 - norm_baseline

delta_blur18_baseline_percent = delta_blur18_baseline.divide(norm_baseline).replace([np.inf, -np.inf], np.nan)
delta_blur12_baseline_percent = delta_blur12_baseline.divide(norm_baseline).replace([np.inf, -np.inf], np.nan)

plot_big_heatmap(delta_blur18_baseline, title="Normalized Shapley Value Difference: 18 Pixel Blur vs Baseline", out_png=f'plots/seq3_shapley_delta_blur18_baseline.png', vmax=1, vmin=-1, annotate=True, annotate_threshold=0.1)

In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import spearmanr, pearsonr
DIMS = ['safe', 'lively', 'beautiful', 'wealthy', 'depressing', 'boring']
CATS = ['building', 'traffic', 'street', 'vegetation', 'sky', 'person']
def safe_corr(x, y, method="spearman", min_n=3):
    x = np.asarray(x).astype(float)
    y = np.asarray(y).astype(float)
    m = np.isfinite(x) & np.isfinite(y)
    x, y = x[m], y[m]
    n = x.size
    if n < min_n:
        return np.nan, n, "too_few_pairs"
    if np.nanstd(x) == 0 and np.nanstd(y) == 0:
        # both constant and equal → perfect agreement in practice
        return 1.0 if np.allclose(x, y) else 0.0, n, "both_constant"
    if np.nanstd(x) == 0 or np.nanstd(y) == 0:
        return np.nan, n, "one_constant"

    if method == "spearman":
        r, _ = spearmanr(x, y)
        if np.isnan(r):  # fallback if ranks tie out
            r, _ = pearsonr(x, y)
            return r, n, "pearson_fallback"
        return r, n, "ok"
    else:
        r, _ = pearsonr(x, y)
        return r, n, "ok"
    
r, n, note = safe_corr(baseline_piv.values.ravel(),
                       blur_18_piv.values.ravel(),
                       method="pearson")
print(f"Global ρ={r:.3f} (n={n}, {note})")

In [ ]:
# compare baseline with 18 px RGB blur
D = len(DIMS)   # 6
C = len(CATS)   # 6

# Ensure perfect alignment of indices/columns first
blur18 = blur_18_piv.reindex_like(baseline_piv)

rows = {}
for i, dim in enumerate(DIMS):
    # columns for this dimension block
    x = baseline_piv[dim].values.ravel()
    y = blur18[dim].values.ravel()
    r, n, note = safe_corr(x/np.amax(abs(x)), y/np.amax(abs(y)), method="spearman")
    p, n, note = safe_corr(x/np.amax(abs(x)), y/np.amax(abs(y)), method="pearson")

    rows[dim] = {"spearman_r":r, "pearson_r": p, "n_pairs": n, "note": note}

corr_by_dim = pd.DataFrame(rows).T
print(corr_by_dim)

In [ ]:
# compare baseline with 12 px RGB blur
D = len(DIMS)   # 6
C = len(CATS)   # 6

# Ensure perfect alignment of indices/columns first
blur12 = blur_12_piv.reindex_like(baseline_piv)

rows = {}
for i, dim in enumerate(DIMS):
    # columns for this dimension block
    c0, c1 = i*C, (i+1)*C
    x = baseline_piv[dim].values.ravel()
    y = blur12[dim].values.ravel()
    r, n, note = safe_corr(x/np.amax(abs(x)), y/np.amax(abs(y)), method="spearman")
    p, n, note = safe_corr(x/np.amax(abs(x)), y/np.amax(abs(y)), method="pearson")
    rows[dim] = {"spearman_r":r, "pearson_r": p, "n_pairs": n, "note": note}

corr_by_dim = pd.DataFrame(rows).T
print(corr_by_dim)

In [ ]:
# compare baseline with solid grey
D = len(DIMS)   # 6        
C = len(CATS)   # 6
# Ensure perfect alignment of indices/columns first
solid_grey = solid_grey_piv.reindex_like(baseline_piv)
rows = {}
for i, dim in enumerate(DIMS):
    # columns for this dimension block
    c0, c1 = i*C, (i+1)*C
    x = baseline_piv[dim].values.ravel()
    y = solid_grey[dim].values.ravel()
    r, n, note = safe_corr(x/np.amax(abs(x)), y/np.amax(abs(y)), method="spearman")
    p, n, note = safe_corr(x/np.amax(abs(x)), y/np.amax(abs(y)), method="pearson")
    rows[dim] = {"spearman_r":r, "pearson_r": p, "n_pairs": n, "note": note}

corr_by_dim = pd.DataFrame(rows).T
print(corr_by_dim)

In [ ]:
# compare 18 px RGB blur vs 12 px RGB blur
D = len(DIMS)   # 6
C = len(CATS)   # 6
# Ensure perfect alignment of indices/columns first
blur12 = blur_12_piv.reindex_like(blur_18_piv)
rows = {}
for i, dim in enumerate(DIMS):
    # columns for this dimension block
    c0, c1 = i*C, (i+1)*C
    x = blur_18_piv[dim].values.ravel()
    y = blur12[dim].values.ravel()
    r, n, note = safe_corr(x/np.amax(abs(x)), y/np.amax(abs(y)), method="spearman")
    p, n, note = safe_corr(x/np.amax(abs(x)), y/np.amax(abs(y)), method="pearson")
    rows[dim] = {"spearman_r":r, "pearson_r": p, "n_pairs": n, "note": note}

corr_by_dim = pd.DataFrame(rows).T
print(corr_by_dim)

In [ ]:
# compare 18 px RGB blur vs solid grey
D = len(DIMS)   # 6
C = len(CATS)   # 6
# Ensure perfect alignment of indices/columns first
solid_grey = solid_grey_piv.reindex_like(blur_18_piv)
rows = {}
for i, dim in enumerate(DIMS):
    # columns for this dimension block
    c0, c1 = i*C, (i+1)*C
    x = blur_18_piv[dim].values.ravel()
    y = solid_grey[dim].values.ravel()
    r, n, note = safe_corr(x/np.amax(abs(x)), y/np.amax(abs(y)), method="spearman")
    p, n, note = safe_corr(x/np.amax(abs(x)), y/np.amax(abs(y)), method="pearson")
    rows[dim] = {"spearman_r":r, "pearson_r": p, "n_pairs": n, "note": note}

corr_by_dim = pd.DataFrame(rows).T
print(corr_by_dim)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from scipy.stats import spearmanr, pearsonr

# your feature colors
feature_styles = {
    "building":  dict(color="#93A3FA", marker="o"),
    "person":    dict(color="#1B2ECC", marker="^"),
    "vegetation":dict(color="#F589A7", marker="s"),
    "street":    dict(color="#E64E5F", marker="D"),
    "traffic":   dict(color="#A8A8A8", marker="P"),
    "sky":       dict(color="#444444", marker="X"),
}

comparisons = [
    (blur_12_piv, baseline_piv, "12 px RGB vs Baseline"),
    (blur_18_piv, baseline_piv, "18 px RGB vs Baseline"),
    (solid_grey_piv, baseline_piv, "Solid Grey vs Baseline"),
    #(solid_white_piv, baseline_piv, "Solid White vs Baseline"),
    (blur_12_piv, blur_18_piv, "12 px RGB vs 18 px RGB"),
    (solid_grey_piv, blur_18_piv, "Solid Grey vs 18 px RGB"),
]

fig, axes = plt.subplots(
    nrows=len(DIMS),
    ncols=len(comparisons),
    figsize=(3.2 * len(comparisons), 2.6 * len(DIMS)),
    sharex=True, sharey=True
)

for i, dim in enumerate(DIMS):
    for j, (test_df, ref_df, label) in enumerate(comparisons):
        ax = axes[i, j]
        all_x, all_y, all_feat = [], [], []

        # Update subplot title with clear Ref/Test indication
        test_label = label.split(" vs ")[0]
        ref_label = label.split(" vs ")[1]
        if i==0:
            ax.set_title(f"{label}\nRef: {ref_label}, Test: {test_label}", fontsize=12)

        for feat, style in feature_styles.items():
            x = ref_df[dim][feat].values
            y = test_df[dim][feat].values
            x = x / (np.nanmax(np.abs(x)) + 1e-9)
            y = y / (np.nanmax(np.abs(y)) + 1e-9)
            all_x.extend(x)
            all_y.extend(y)
            sns.scatterplot(
                x=x, y=y, s=18, edgecolor="k",
                alpha=0.7 if feat in ["building","street"] else 0.8, ax=ax, legend=False,
                **style
            )

        # diagonal reference line
        ax.axline((0, 0), slope=1, color="k", lw=0.8, ls="--")

        # compute Spearman r
        mask = np.isfinite(all_x) & np.isfinite(all_y)
        rho, _ = pearsonr(np.array(all_x)[mask], np.array(all_y)[mask])
        ax.text(
            0.05, 0.92, f"ρ={rho:.2f}",
            transform=ax.transAxes, ha="left", va="top",
            fontsize=16, bbox=dict(boxstyle="round,pad=0.2", fc="white", alpha=0.6)
        )

        # Axis labels
        if j == 0:
            ax.set_ylabel(f"{dim}\n$\\hat{{\\phi}}_{{ref}}$", fontsize=16)
        else:
            ax.set_ylabel(None)

        if i == len(DIMS) - 1:
            ax.set_xlabel("$\\hat{{\\phi}}_{{test}}$", fontsize=16)
        else:
            ax.set_xlabel(None)

# shared xlabel
#for ax in axes[-1]:
#    ax.set_xlabel("Reference Shapley", fontsize=16)

legend_handles = [
    plt.Line2D([0], [0],
        marker=feature_styles[f]["marker"],
        color='none',
        markerfacecolor=feature_styles[f]["color"],
        markersize=6,
        label=f
    )
    for f in feature_styles
]
fig.legend(
    handles=legend_handles,
    loc="lower center",
    ncol=3,
    bbox_to_anchor=(0.5, -0.05),
    title="Feature",
    fontsize=9, title_fontsize=10
)

fig.suptitle("Pairwise stability across masking regimes", fontsize=11)
fig.tight_layout(rect=[0, 0, 0.98, 0.96])
plt.show()

In [ ]:
fig, ax = plt.subplots(3,2, figsize=(6,8))
dimensions = ['safe', 'lively', 'beautiful', 'wealthy', 'boring', 'depressing']
for i, d in enumerate(dimensions):
    r = i//2
    c = i%2
    
    dim_df = full_df[full_df.dimension==d]
    ax[r][c].scatter(dim_df.phi, dim_df.confidence)
    ax[r][c].set_title(d.capitalize())
    ax[r][c].set_xlabel('$\phi$')
    ax[r][c].set_ylabel('Confidence')

fig.tight_layout()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# --- config ---
CATS = ["building","traffic","person","vegetation","street","sky"]
DIM_ORDER = ["safe","lively","beautiful","wealthy","boring","depressing"]  # adjust if needed
req_cols = {"image_id","dimension","feature","phi","confidence"}

# --- basic checks ---
missing = req_cols - set(full_df.columns)
if missing:
    raise ValueError(f"full_df is missing required columns: {sorted(missing)}")

# keep just what we need and drop unusable rows
df = (
    full_df.loc[:, ["image_id","dimension","feature","phi","confidence"]]
            .dropna(subset=["phi","confidence"])
            .copy()
)
# ensure consistent types/case
df["dimension"] = df["dimension"].astype(str).str.lower()
df["feature"]  = df["feature"].astype(str).str.lower()

# --- pick the feature with the largest |phi| per (image, dimension) ---
# For each group, take the row with max absolute phi
idx = df.groupby(["image_id","dimension"])["phi"].apply(lambda s: s.abs().idxmax())
phi_max_df = df.loc[idx].copy()
phi_sum_df = df.loc[idx].copy()

# Rename for clarity and carry along the absolute value for the x-axis
phi_max_df = phi_max_df.rename(columns={"phi":"phi_max", "feature":"max_feature"})
phi_max_df["abs_phi_max"] = phi_max_df["phi_max"].abs()
phi_sum_df = phi_sum_df.rename(columns={"phi":"phi_sum"})

# optional: order dimensions nicely if all present
dims = [d for d in DIM_ORDER if d in set(phi_max_df["dimension"])]
if not dims:
    dims = sorted(phi_max_df["dimension"].unique())

# --- plotting ---
fig, axes = plt.subplots(2, 3, figsize=(11, 8), sharex=True, sharey=True)
axes = axes.ravel()

# simple categorical color map (matplotlib default cycle is fine; mapping for consistency)
from itertools import cycle
color_cycle = cycle(plt.rcParams['axes.prop_cycle'].by_key()['color'])
cat_colors = {c: next(color_cycle) for c in CATS}

for ax, d in zip(axes, dims):
    sub = phi_max_df[phi_max_df["dimension"] == d]
    # plot each winning feature with its own color
    for cat in CATS:
        mask = (sub["max_feature"] == cat)
        if mask.any():
            ax.scatter(
                sub.loc[mask, "abs_phi_max"],
                sub.loc[mask, "confidence"],
                s=18, alpha=0.2, label=cat, c=cat_colors[cat],
            )
    ax.set_title(d.capitalize())
    ax.set_xlabel(r"max $|\phi|$ across categories")
    ax.set_ylabel("Confidence")

# tidy legend: one shared legend outside the grid
handles, labels = axes[0].get_legend_handles_labels()
by_label = dict(zip(labels, handles))
fig.legend(
    by_label.values(), by_label.keys(),
    loc="lower center", ncol=6, frameon=False, bbox_to_anchor=(0.5, -0.01)
)

# layout that won't crop when saving
plt.tight_layout()
plt.subplots_adjust(bottom=0.12)  # leave room for legend
# plt.savefig("phi_max_vs_confidence_by_dimension.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
full_df['image_uuid'] = full_df['seq_id'] + '::' + full_df['image_id']

In [ ]:
phi_conf_data = []


for uid, img_data in full_df.groupby('image_uuid'):
    
    for d, dim_data in img_data.groupby('dimension'):
        # max shapley score
        max_phi = dim_data.phi.abs().max()
        max_phi_idx = dim_data.phi.abs().idxmax()
        # feature of max phi
        max_feat = dim_data.loc[max_phi_idx]['feature']
        # sum phi
        sum_phi = dim_data.phi.abs().sum()
        # conf will always be the same for all (img, dim) pairs
        conf = dim_data.confidence.iloc[0]
        # add to data
        row = {'image_uid':uid,
               'dimension':d,
               'max_feature':max_feat,
               'max_phi':max_phi,
               'sum_phi':sum_phi,
               'confidence':conf}
        phi_conf_data.append(row)

phi_conf_df = pd.DataFrame(phi_conf_data)

In [ ]:

fig, ax = plt.subplots(3,2, figsize=(8,8))

for i, dim in enumerate(dimensions):
    
    r = i//2
    c = i%2
    
    dim_data = phi_conf_df[phi_conf_df.dimension==dim]
    
    ax[r][c].scatter(dim_data.max_phi, dim_data.confidence)
    ax[r][c].set_title(dim.capitalize())
    
    ax[r][c].set_xlabel('max$(|\phi|)$')
    ax[r][c].set_ylabel('Confidence')
    
fig.tight_layout()
    
#plt.scatter(phi_conf_df.sum_phi, phi_conf_df.confidence)

In [ ]:
fig, ax = plt.subplots(3,2, figsize=(8,8))

for i, dim in enumerate(dimensions):
    
    r = i//2
    c = i%2
    
    dim_data = phi_conf_df[phi_conf_df.dimension==dim]
    
    ax[r][c].scatter(dim_data.sum_phi, dim_data.confidence)
    ax[r][c].set_title(dim.capitalize())
    
    ax[r][c].set_xlabel('$\sum(|\phi|)$')
    ax[r][c].set_ylabel('Confidence')
    
fig.tight_layout()

In [ ]:
stats = (full_df.groupby(["seq_id","feature","dimension"], observed=True)["phi"]
            .agg(mu="mean", sigma=lambda s: s.std(ddof=1))
            .reset_index())
dfn = full_df.merge(stats, on=["seq_id","feature","dimension"], how="left")
eps = 1e-9
den = dfn["sigma"].where(dfn["sigma"].notna() & (dfn["sigma"] > 0), eps)
dfn["z"] = (dfn["phi"] - dfn["mu"]) / den

In [ ]:
full_df.groupby(["seq_id","feature","dimension"])['phi'].agg(mu="mean", sigma=lambda s: s.std(ddof=1))

# Top feature allignment

In [ ]:
# ensure a unique per-image id
full_df = full_df.copy()
full_df['image_uuid'] = full_df['seq_id'].astype(str) + '-' + full_df['image_id'].astype(str)

dimensions = ['safe', 'lively', 'beautiful', 'wealthy', 'boring', 'depressing']
df = full_df[full_df['dimension'].isin(dimensions)].copy()

# optional: if phi can be NaN, drop or impute
df = df[pd.notna(df['phi'])]

def topk_flags(s: pd.Series, k: int) -> pd.Series:
    """Boolean: True if value is in top-k of the group (ties included)."""
    if s.empty:
        return s.astype(bool)
    # kth largest threshold (ties included)
    kth = s.nlargest(min(k, len(s))).min()
    return s >= kth

def bottomk_flags(s: pd.Series, k: int) -> pd.Series:
    """Boolean: True if value is in bottom-k of the group (ties included)."""
    if s.empty:
        return s.astype(bool)
    kth = s.nsmallest(min(k, len(s))).max()
    return s <= kth

# Compute flags per (image_uuid, dimension)
grp = ['image_uuid', 'dimension']
df['top1']  = df.groupby(grp)['phi'].transform(lambda s: topk_flags(s, 1))
df['top2']  = df.groupby(grp)['phi'].transform(lambda s: topk_flags(s, 2))
df['bot1']  = df.groupby(grp)['phi'].transform(lambda s: bottomk_flags(s, 1))
df['bot2']  = df.groupby(grp)['phi'].transform(lambda s: bottomk_flags(s, 2))

# Alignment metrics by dimension
pp = df[df['m_pc'] == 'PP']
np_ = df[df['m_pc'] == 'NP']

top1_alignment = pp.groupby('dimension')['top1'].mean().to_dict()   # proportion of PP in top-1
top2_alignment = pp.groupby('dimension')['top2'].mean().to_dict()   # proportion of PP in top-2
bot1_alignment = np_.groupby('dimension')['bot1'].mean().to_dict()  # proportion of NP in bottom-1
bot2_alignment = np_.groupby('dimension')['bot2'].mean().to_dict()  # proportion of NP in bottom-2

# (Optional) also keep denominators so you can report n and CIs
top_denoms = pp.groupby('dimension').size().to_dict()
bot_denoms = np_.groupby('dimension').size().to_dict()
        
        
    


In [ ]:
from scipy.stats import ttest_ind
pp_depr = pp[pp.dimension=='depressing']['top2']

depr_sig = {}
for d in dims:
    pp_dim = pp[pp.dimension==d]['top2']
    sig = ttest_ind(pp_depr, pp_dim).pvalue
    depr_sig[d] = sig

depr_sig
    

In [ ]:
print(top1_alignment)
print(top2_alignment)
print(bot1_alignment)
print(bot2_alignment)

In [ ]:
top_denoms

In [ ]:
bot_denoms

In [ ]:
def weighted_average(scores, counts):
    numer = 0
    denom = 0
    for d, score in scores.items():
        w_score = score*counts[d]
        numer+=w_score
        denom+=counts[d]
    return numer/denom

In [ ]:
print(weighted_average(top1_alignment, top_denoms))
print(weighted_average(top2_alignment, top_denoms))
print(weighted_average(bot1_alignment, bot_denoms))
print(weighted_average(bot2_alignment, bot_denoms))

In [ ]:
top1_alignment

In [ ]:
# get the number of instances in which the Shapley score for boring/building is positive vs negative
boring_building_positive = full_df[(full_df.dimension=='boring') & (full_df.feature=='building') & (full_df.phi > 0)]
boring_building_negative = full_df[(full_df.dimension=='boring') & (full_df.feature=='building') & (full_df.phi < 0)]

# get the number of instances in which m_pc is PP/NP for boring building
boring_building_pp = full_df[(full_df.dimension=='boring') & (full_df.feature=='building') & (full_df.m_pc == 'PP')]
boring_building_np = full_df[(full_df.dimension=='boring') & (full_df.feature=='building') & (full_df.m_pc == 'NP')]

print(f'phi(boring, building)>0 : {len(boring_building_positive)}')
print(f'phi(boring, building)<0 : {len(boring_building_negative)}')

print(f'm_pc=PP : {len(boring_building_pp)}')
print(f'm_pc=NP : {len(boring_building_np)}')

In [ ]:
# get the number of instances in which the Shapley score for beautiful/building is positive vs negative
beautiful_building_positive = full_df[(full_df.dimension=='beautiful') & (full_df.feature=='building') & (full_df.phi > 0)]
beautiful_building_negative = full_df[(full_df.dimension=='beautiful') & (full_df.feature=='building') & (full_df.phi < 0)]

# get the number of instances in which m_pc is PP/NP for beautiful building
beautiful_building_pp = full_df[(full_df.dimension=='beautiful') & (full_df.feature=='building') & (full_df.m_pc == 'PP')]
beautiful_building_np = full_df[(full_df.dimension=='beautiful') & (full_df.feature=='building') & (full_df.m_pc == 'NP')]

print(f'phi(beautiful, building)>0 : {len(beautiful_building_positive)}')
print(f'phi(beautiful, building)<0 : {len(beautiful_building_negative)}')

print(f'm_pc=PP : {len(beautiful_building_pp)}')
print(f'm_pc=NP : {len(beautiful_building_np)}')

In [ ]:
# get the number of instances in which the Shapley score for depressing/building is positive vs negative
depressing_building_positive = full_df[(full_df.dimension=='depressing') & (full_df.feature=='building') & (full_df.phi > 0)]
depressing_building_negative = full_df[(full_df.dimension=='depressing') & (full_df.feature=='building') & (full_df.phi < 0)]

# get the number of instances in which m_pc is PP/NP for depressing building
depressing_building_pp = full_df[(full_df.dimension=='depressing') & (full_df.feature=='building') & (full_df.m_pc == 'PP')]
depressing_building_np = full_df[(full_df.dimension=='depressing') & (full_df.feature=='building') & (full_df.m_pc == 'NP')]

print(f'phi(depressing, building)>0 : {len(depressing_building_positive)}')
print(f'phi(depressing, building)<0 : {len(depressing_building_negative)}')

print(f'm_pc=PP : {len(depressing_building_pp)}')
print(f'm_pc=NP : {len(depressing_building_np)}')

In [ ]:
df